# Multilingual Fine-Tuning for Legal Domain (mBART)
Demonstrates fine-tuning `facebook/mbart-large-50-many-to-many-mmt` on 50 English-Hindi legal pairs, and evaluating on 15 pairs logging BLEU scores.
As requested, this notebook functions even if fine-tuning is skipped (e.g., due to local compute constraints), by evaluating the base model instead.

In [ ]:
!pip install -q transformers datasets evaluate sacrebleu sentencepiece torch torchtext

In [ ]:
import json
from datasets import Dataset, DatasetDict

# Load the dataset
with open('legal_pairs.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

dataset = DatasetDict({
    'train': Dataset.from_list(data['train']),
    'validation': Dataset.from_list(data['validation'])
})
print("Dataset loaded:", dataset)

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model_name = "facebook/mbart-large-50-many-to-many-mmt"
try:
    tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
    model = MBartForConditionalGeneration.from_pretrained(model_name)
    print("Model loaded successfully.")
except Exception as e:
    print(f"Could not load model (check internet or memory): {e}")

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import torch

# 1. Ensure GPU is used if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training will run on: {device}")

# 2. Configure training parameters optimized for Colab limits
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=1,  # Lower batch size to prevent Colab Memory OOM
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=(device == "cuda"),  # Automatically use mixed precision if GPU is enabled
)

# 3. Modern approach to Seq2Seq Tokenization
def preprocess_function(examples):
    inputs = [ex for ex in examples['en']]
    targets = [ex for ex in examples['hi']]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
    return model_inputs
    
tokenized_datasets = dataset.map(preprocess_function, batched=True)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 4. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting fine-tuning...")
trainer.train()
print("Training complete! Saving model...")

# 5. Save the final model explicitly
OUTPUT_PATH = "/content/legal-mbart-finetuned"
trainer.save_model(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)
print(f"\nSUCCESS! Model and tokenizer saved to {OUTPUT_PATH}")


In [ ]:
import evaluate
sacrebleu = evaluate.load("sacrebleu")

def evaluate_model(model_eval, tokenizer_eval, eval_data):
    model_eval.eval()
    tokenizer_eval.src_lang = "en_XX"
    preds = []
    refs = []
    for item in eval_data:
        inputs = tokenizer_eval(item['en'], return_tensors="pt")
        
        try:
            device = next(model_eval.parameters()).device
        except:
            device = "cpu"
            
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        generated_tokens = model_eval.generate(
            **inputs,
            forced_bos_token_id=tokenizer_eval.lang_code_to_id["hi_IN"],
            max_length=50
        )
        
        pred_text = tokenizer_eval.batch_decode(generated_tokens, skip_special_tokens=True)[0]
        preds.append(pred_text)
        refs.append([item['hi']])
        
    results = sacrebleu.compute(predictions=preds, references=refs)
    return results['score']



In [ ]:
# Only evaluate to demonstrate it works regardless of fine-tuning.
print("Evaluating BLEU score on validation set...")
try:
    bleu_score = evaluate_model(model, tokenizer, dataset['validation'])
    print(f"Validation BLEU Score (Pre-trained/Fine-tuned fallback): {bleu_score:.2f}")
except Exception as e:
    print(f"Evaluation skipped due to error: {e}")